# iota — Kaggle run (train → eval → profile → plot → publish)

Every stage publishes its results to the **`kaggle-results`** branch of
`RohanBanerjee88/iota` (under `runs/<RUN_ID>/`), so the results reach the repo — and
Claude — without copy-pasting. Weights go to the HF Hub.

**One-time setup (Kaggle notebook sidebar):**
1. *Settings → Accelerator →* **GPU T4 x2** (the code uses one GPU).
2. *Settings →* **Internet on** (needs a phone-verified Kaggle account).
3. *Add-ons → Secrets →* add **`GH_TOKEN`** (GitHub fine-grained token, repo `RohanBanerjee88/iota`,
   *Contents: Read and write*) and **`HF_TOKEN`** (Hugging Face token with *Write*). Tick both for this notebook.

**Order of runs** (set `PLAN` in cell 1):
- `"smoke"` — interactive, ~10 min. Proves GPU, tokens, and the whole pipeline. Must end with `GitHub link: OK`.
- `"tune"` — a short, identical lr probe for all three (*Save & Run All*, ~3–3.5 h). Claude sets each lr from it.
- `"A"` — train all three + sanity gate + eval passes 1 & 3. Use *Save Version → Save & Run All* so it runs
  unattended.
- `"B"` — eval pass 2 + cost profile + the figure. Also *Save & Run All*.
- `"all"` — A and B in one session, if it fits.

Every stage is resumable: re-running skips finished models, so a timeout only costs the model in progress.

In [ ]:
# ---- 1. CONFIG (edit me) ---------------------------------------------------
RUN_ID  = "r02"                       # one id per experiment -- reuse it for plan A and plan B
PLAN    = "smoke"                     # "smoke" -> "tune" -> "A" -> "B"
BRANCH  = "main"                      # which code branch to run
HF_REPO = "BanerjeeRohan44/iota-r02"  # weights backup; a NEW repo name = a truly fresh start
GH_REPO = "RohanBanerjee88/iota"      # results land on its kaggle-results branch
# ----------------------------------------------------------------------------

STEPS = {
    "smoke": ["tests", "smoke"],
    "tune":  ["tests", "smoke", "tune"],      # short identical lr probe for all three (~3-3.5 h)
    "A":     ["tests", "smoke", "train", "sanity", "eval13"],
    "B":     ["tests", "smoke", "eval2", "profile", "plot"],
    "all":   ["tests", "smoke", "train", "sanity", "eval13", "eval2", "profile", "plot"],
}[PLAN]
print(f"run {RUN_ID!r}, plan {PLAN!r}:", " -> ".join(STEPS))

In [ ]:
# ---- 2. GPU, secrets, helper ------------------------------------------------
import os, subprocess
import torch

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU T4 x2"
print("GPU:", torch.cuda.get_device_name(0), "| torch", torch.__version__)

from kaggle_secrets import UserSecretsClient
_secrets = UserSecretsClient()
for key in ("GH_TOKEN", "HF_TOKEN"):
    try:
        os.environ[key] = _secrets.get_secret(key)
    except Exception:
        raise RuntimeError(f"Secret {key} missing: Add-ons -> Secrets -> add {key}, tick it for this notebook")
    print(f"{key}: loaded")

from huggingface_hub import HfApi
print("HF user:", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"], "| HF repo:", HF_REPO)

os.environ["IOTA_RUN_ID"] = RUN_ID
os.environ["PYTHONUNBUFFERED"] = "1"


def run(cmd, check=True):
    """Run a shell command, streaming output live. Stops the notebook on failure."""
    print(f"\n$ {cmd}", flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0 and check:
        raise RuntimeError(f"FAILED (exit {p.returncode}): {cmd}")
    return p.returncode

def want(step):
    """True if this PLAN includes `step`; otherwise say so, so a skipped cell is never silent."""
    if step in STEPS:
        return True
    print(f"-- skipped '{step}': PLAN={PLAN!r} runs only {STEPS}. "
          f"Change PLAN in cell 1 (e.g. PLAN=\"A\") to run it.")
    return False

In [ ]:
# ---- 3. Get the code --------------------------------------------------------
os.chdir("/kaggle/working")
if not os.path.isdir("iota/.git"):
    run(f"git clone --quiet https://github.com/{GH_REPO}.git iota")
os.chdir("/kaggle/working/iota")
run(f"git fetch --quiet origin {BRANCH} && git checkout --quiet -B {BRANCH} origin/{BRANCH}")
run("git log -1 --format='code: %h %s'")
assert os.path.exists("scripts/sync_results.py"), (
    f"branch {BRANCH!r} predates the Kaggle->GitHub link -- merge the PR or set BRANCH")
# torch / numpy / matplotlib come preinstalled; never reinstall torch on Kaggle.
run("pip install -q safetensors huggingface_hub pyyaml tqdm pytest")

In [ ]:
# ---- 4. Unit tests (~1 min) -------------------------------------------------
if want("tests"):
    run("python -m pytest -q")

In [ ]:
# ---- 5. Smoke: the WHOLE pipeline in ~5-10 min (numbers are meaningless) -----
# Trains 60 steps per model, evals every pass at n=8, profiles, plots, and
# publishes to runs/<RUN_ID>-smoke. Stops the notebook if the GitHub link fails,
# so a bad token costs 10 minutes, not a 9-hour run.
if want("smoke"):
    run("python -u -m scripts.run_all --stage all --smoke")

In [ ]:
# ---- 5b. LR probe: every arch x the same lr grid, same short budget ---------
# Writes tune.csv (published to GitHub after every probe). Claude reads it and
# sets each arch's lr in configs/sweep_*.yaml before plan "A".
if want("tune"):
    run(f"python -u -m scripts.run_all --stage tune --repo {HF_REPO}")

In [ ]:
# ---- 6. Train all three (the long one) --------------------------------------
# Order: hybrid -> transformer -> gated_linear. Each model is backed up to HF and
# published to GitHub the moment it finishes. Watch the per-mode line:
#   [sweep] ... bal 0.71 [assoc 0.83 state 0.59] ...
# assoc AND state must both climb well above 0.01.
if want("train"):
    run(f"python -u -m scripts.run_all --stage train --repo {HF_REPO}")

In [ ]:
# ---- 7. Sanity gate + eval pass 1 (capacity) & pass 3 (control) --------------
if want("sanity"):
    run(f"python -u -m scripts.run_all --stage sanity --repo {HF_REPO}")
if want("eval13"):
    run(f"python -u -m scripts.run_all --stage eval --passes 1,3 --repo {HF_REPO}")

In [ ]:
# ---- 8. Eval pass 2 (length generalisation, up to 8192 tokens) --------------
# Pulls the checkpoints from HF if this is a fresh session. On OOM the eval batch
# halves down to 1 before a cell is recorded as OOM.
if want("eval2"):
    run(f"python -u -m scripts.run_all --stage eval --passes 2 --repo {HF_REPO}")

In [ ]:
# ---- 9. Cost profile + the money figure -------------------------------------
if want("profile"):
    run(f"python -u -m scripts.run_all --stage profile --repo {HF_REPO}")
if want("plot"):
    run(f"python -u -m scripts.run_all --stage plot --repo {HF_REPO}")

In [ ]:
# ---- 10. Look at what we got ------------------------------------------------
from IPython.display import Image, Markdown, display

res = "experiments/results"
run("python -m scripts.run_all --stage status --no-gh --no-hf", check=False)
if os.path.exists(f"{res}/money_figure.png"):
    display(Image(f"{res}/money_figure.png"))
out = subprocess.run(["python", "-m", "scripts.sync_results", "--summary-only", "--run-id", RUN_ID],
                     capture_output=True, text=True).stdout
display(Markdown(out))
print(f"Published: https://github.com/{GH_REPO}/tree/kaggle-results/runs/{RUN_ID}")